# Topic embeddings with SGNS

Learn topic embeddings from the top 1,000 leaf topics selected in the previous notebook. In [word2vecf](https://github.com/BIU-NLP/word2vecf), topics are **words** and authors are **contexts**. Each retained comment contributes one topic–author pair, including repeated pairs. The comment text isn't used.

The current settings are 150 dimensions, 35 negative samples, a starting learning rate of 0.18, subsampling at 0.0043, and one pass with four threads. Negative authors are sampled based on their comment counts raised to 0.75. Subsampling applies to both topics and authors.

Comments are read in chunks and the pairs are shuffled globally before training. `SHUFFLE_SEED` controls that shuffle; it doesn't control the trainer's weight initialization. Training with multiple threads can still give slightly different results between runs.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from collections import Counter
import json
import subprocess
import numpy as np
import pandas as pd

DATA_DIR = Path("my_data")
RAW_DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "sgns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
USE_RAW_COMMENTS = False
TOOL_DIR = Path(".tools/word2vecf")
TOOL_COMMIT = "2d503b095546660e0bb8b0594c7b442c7c533e49"
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
CHUNK_SIZE = 250_000
SHUFFLE_SEED = 42

# word2vecf hyperparameters
DIMENSIONS = 150
NEGATIVE = 35
ALPHA = 0.18
SAMPLE = 0.0043 # subsampling threshold
EPOCHS = 1
THREADS = 4 

## 1. Build word2vecf

Clone word2vecf if it isn't already in `.tools/word2vecf`, check that it matches `TOOL_COMMIT`, and compile it with `cc`. This needs Git and a C compiler. Compiler output and errors are saved to `my_data/sgns/build.log`.


In [2]:
# Clone word2vecf repo if the tool has not already been downloaded
if not TOOL_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/BIU-NLP/word2vecf.git", str(TOOL_DIR)], check=True)
    subprocess.run(["git", "-C", str(TOOL_DIR), "checkout", TOOL_COMMIT], check=True)

# Check that the tool is at the right commit for reproducibility
actual_commit = subprocess.check_output(["git", "-C", str(TOOL_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != TOOL_COMMIT:
    raise ValueError("word2vecf version differs from the pinned implementation")

# Compile the word2vecf tool
trainer = (TOOL_DIR / "word2vecf").resolve() # absolute path
with (OUTPUT_DIR / "build.log").open("w") as log:
    subprocess.run(["cc", "-O3", "-pthread", *[str(TOOL_DIR / name) for name in
                    ("word2vecf.c", "vocab.c", "io.c")], "-lm", "-o", str(trainer)],
                   stdout=log, stderr=subprocess.STDOUT, check=True)

## 2. Load selected topics, articles and comments

Read `topics_top_1000.csv` and `articles_top_1000.csv` from `my_data` and strip whitespace from the topic labels. 

`comment_chunks()` reads  `comments_top_1000`, in batches of 250,000 rows. With `USE_RAW_COMMENTS = False`, it reads `my_data/comments_top_1000.csv`. Setting it to `True` reads the original comment TSV files in `data` instead.


In [3]:
# Load topics
topics_df = pd.read_csv(DATA_DIR / "topics_top_1000.csv", keep_default_na=False,
                              dtype={column: "string" for column in TOPIC_COLUMNS})
topics_df[TOPIC_COLUMNS] = topics_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())

# Load articles and create a mapping from article ID to topic ID
articles_df = pd.read_csv(DATA_DIR / "articles_top_1000.csv", dtype="string",
                          keep_default_na=False, usecols=["Article_Id", "Topic_Id", *TOPIC_COLUMNS])
articles_df[TOPIC_COLUMNS] = articles_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())
article_to_topic = (articles_df.set_index("Article_Id")["Topic_Id"])

# Comment chunks generator
def comment_chunks():
    if USE_RAW_COMMENTS:
        paths = sorted(RAW_DATA_DIR.glob("comments_file_*_[0-9][0-9].tsv"))
        if not paths:
            raise FileNotFoundError("No raw comment shards found")
    else:
        paths = [DATA_DIR / "comments_top_1000.csv"]
    for path in paths:
        with pd.read_csv(path, sep="\t" if USE_RAW_COMMENTS else ",", dtype="string",
                         keep_default_na=False, usecols=["Article_Id", "Author_Id"],
                         chunksize=CHUNK_SIZE) as reader:
            yield from reader

## 3. Create the pairs, shuffle, and train

The trainer writes the topic vectors to `topic_vectors.txt` and its output and errors to `training.log`. `training_stats` keeps the pair count, number of authors, excluded blank-author count, and comment counts per topic for the metadata saved below. These counts are measured before the trainer's subsampling. The author-code mapping is returned as `user_codes` and kept in memory to identify the exported user vectors.


In [4]:
def train_embeddings():
    """
    Train topic and user embeddings from comments supplied by comment_chunks().

    Temporary files in OUTPUT_DIR/sgns-*/ (automatically cleaned up):
      pairs.bin: binary pairs used for shuffling; not a trainer input.
      pairs.txt: shuffled topic-author pairs; trainer input (-train).
      topics.vocab: topic tokens/counts; trainer input (-wvocab).
      users.vocab: author tokens/counts; trainer input (-cvocab).

    Persistent files in OUTPUT_DIR:
      topic_vectors.txt: learned topic vectors written by the trainer.
      user_vectors.txt: learned author/context vectors written by the trainer.
      training.log: trainer stdout and stderr.
      
    Returns training counts and user_codes, kept in memory for the CSV exports.
    Embeddings are saved to the vector files above.
    """
    rng = np.random.default_rng(SHUFFLE_SEED)
    user_codes = {} # maps author IDs to int codes for the trainer (more efficient)
    user_counts = Counter() # maps user codes to comment counts for the vocab file
    topic_counts = np.zeros(len(topics_df), dtype=np.int64) # counts comments per topic for the vocab file
    total_pairs = blank_authors = 0
    with TemporaryDirectory(prefix="sgns-", dir=OUTPUT_DIR) as scratch:
        # Create a temporary binary file (pairs.bin) of (topic_id, user_code) pairs, one per valid comment (read by chunks)
        scratch = Path(scratch)
        binary_pairs = scratch / "pairs.bin"
        with binary_pairs.open("wb") as output:
            for chunk in comment_chunks():
                # Raw shards also contain articles outside the selected topics
                topic_ids = chunk["Article_Id"].map(article_to_topic) 
                chunk = chunk.loc[topic_ids.notna()]
                topic_ids = topic_ids.dropna()

                # Check for blank authors and count them, then filter them out
                valid = chunk["Author_Id"].str.strip().ne("")
                blank_authors += int((~valid).sum())
                authors = chunk.loc[valid, "Author_Id"]
                topic_ids = topic_ids.loc[valid].to_numpy(dtype=np.int32)

                # Map authors to user codes for efficient training and shuffling
                for author in authors.unique():
                    if author not in user_codes:
                        user_codes[author] = len(user_codes)
                users = authors.map(user_codes).to_numpy(dtype=np.int32)

                # Write the (topic, author) pairs to the binary file and update counts
                np.column_stack((topic_ids, users)).astype(np.int32).tofile(output)

                # Update counts for topics and users
                topic_counts += np.bincount(topic_ids, minlength=len(topics_df))
                ids, counts = np.unique(users, return_counts=True)
                user_counts.update(dict(zip(ids.tolist(), counts.tolist())))
                total_pairs += len(users)

        n_users = len(user_codes)

        # Access pais.bin as a memory-mapped array and shuffle the pairs
        pairs = np.memmap(binary_pairs, dtype=np.int32, mode="r+", shape=(total_pairs, 2))
        rng.shuffle(np.asarray(pairs).view(np.uint64).reshape(-1))
        pairs.flush()

        # The trainer expects a txt file so we write pairs.bin to pairs.txt
        with (scratch / "pairs.txt").open("w") as output:
            output.write("\n") # first line is skipped by the trainer
            for start in range(0, total_pairs, CHUNK_SIZE):
                # Pairs writtern as: t<topic_id> u<user_code> for convenience
                output.writelines(f"t{topic} u{user}\n" for topic, user in np.asarray(pairs[start:start + CHUNK_SIZE]))
        del pairs

        # Write vocab files for topics and users
        with (scratch / "topics.vocab").open("w") as output:
            output.writelines(f"t{i} {count}\n" for i, count in enumerate(topic_counts))
        with (scratch / "users.vocab").open("w") as output:
            output.writelines(f"u{i} {user_counts[i]}\n" for i in range(n_users))

        # Train the embeddings using the word2vecf tool
        command = [str(trainer), "-train", "pairs.txt", "-wvocab", "topics.vocab",
                   "-cvocab", "users.vocab", "-output", str((OUTPUT_DIR / "topic_vectors.txt").resolve()),
                   "-dumpcv", str((OUTPUT_DIR / "user_vectors.txt").resolve()),
                   "-size", str(DIMENSIONS), "-negative", str(NEGATIVE), "-alpha", str(ALPHA),
                   "-sample", str(SAMPLE), "-iters", str(EPOCHS), "-threads", str(THREADS),
                   "-binary", "0", "-debug", "0"]
        with (OUTPUT_DIR / "training.log").open("w") as log:
            subprocess.run(command, cwd=scratch, stdout=log, stderr=subprocess.STDOUT, check=True)
    return dict(training_pairs=total_pairs, users=n_users, excluded_blank_authors=blank_authors,
                topic_comment_counts=topic_counts.tolist()), user_codes

training_stats, user_codes = train_embeddings()

## 4. Load and save the topic and user embeddings

This section writes five files to `my_data/sgns`:

- `topic_embeddings.csv`: topic IDs, topic labels, and the 150 raw embedding coordinates.
- `topic_embeddings_normalized.csv`: the same IDs and labels with unit-length vectors.
- `user_embeddings.csv`: original author IDs, user tokens, and raw user embedding coordinates.
- `user_embeddings_normalized.csv`: the same authors with unit-length vectors (zero vectors stay zero).
- `run_metadata.json`: hyperparameters, shuffle seed, selection description, and training counts.

The earlier sections also leave these files in the same folder:

- `topic_vectors.txt`: the trainer's original topic-vector output, read by this section.
- `user_vectors.txt`: author/context vectors exported by the trainer.
- `build.log`: compiler output and errors.
- `training.log`: trainer output and errors.


In [5]:
# Load the learned topic vectors and save them to a CSV file along with the topic metadata
vectors_by_token = {}
with (OUTPUT_DIR / "topic_vectors.txt").open() as source:
    rows, dimensions = map(int, source.readline().split())
    for line in source:
        token, *values = line.split()
        vectors_by_token[token] = np.asarray(values, dtype=np.float32)
topic_embeddings = np.stack([vectors_by_token[f"t{i}"] for i in topics_df["Topic_Id"]])
topic_embeddings_df = pd.concat([
    topics_df[["Topic_Id", *TOPIC_COLUMNS]],
    pd.DataFrame(topic_embeddings, columns=[f"embedding_{i:03d}" for i in range(DIMENSIONS)])
], axis=1)
topic_embeddings_df.to_csv(OUTPUT_DIR / "topic_embeddings.csv", index=False)


# Normalize the topic embeddings and save them to a CSV file
norms = np.linalg.norm(topic_embeddings, axis=1, keepdims=True)
normalized_topic_embeddings = topic_embeddings / norms
normalized_topic_embeddings_df = pd.concat([
    topics_df[["Topic_Id", *TOPIC_COLUMNS]],
    pd.DataFrame(normalized_topic_embeddings, columns=[f"embedding_{i:03d}" for i in range(DIMENSIONS)])
], axis=1)
normalized_topic_embeddings_df.to_csv(OUTPUT_DIR / "topic_embeddings_normalized.csv", index=False)

# Load user vectors and match them to the original author IDs
users_df = pd.DataFrame({
    "User_Token": [f"u{code}" for code in user_codes.values()],
    "Author_Id": list(user_codes),
})
user_vectors_by_token = {}
with (OUTPUT_DIR / "user_vectors.txt").open() as source:
    rows, dimensions = map(int, source.readline().split())
    for line in source:
        token, *values = line.split()
        user_vectors_by_token[token] = np.asarray(values, dtype=np.float32)
user_embeddings = np.stack([user_vectors_by_token[token] for token in users_df["User_Token"]])
user_embeddings_df = pd.concat(
    [users_df, pd.DataFrame(user_embeddings, columns=[f"embedding_{i:03d}" for i in range(DIMENSIONS)])
], axis=1)
user_embeddings_df.to_csv(OUTPUT_DIR / "user_embeddings.csv", index=False)

# Normalize user vectors, leave any untrained zero vectors at zero
user_norms = np.linalg.norm(user_embeddings, axis=1, keepdims=True)
normalized_user_embeddings = np.divide(
    user_embeddings, user_norms, out=np.zeros_like(user_embeddings), where=user_norms != 0
)
normalized_user_embeddings_df = pd.concat([
    users_df, pd.DataFrame(normalized_user_embeddings, columns=[f"embedding_{i:03d}" for i in range(DIMENSIONS)])
], axis=1)
normalized_user_embeddings_df.to_csv(OUTPUT_DIR / "user_embeddings_normalized.csv", index=False)

# Save run metadata to a JSON file for reproducibility
run_metadata = dict(
    dimensions=DIMENSIONS, negative=NEGATIVE, alpha=ALPHA, sample=SAMPLE,
    epochs=EPOCHS, threads=THREADS, shuffle_seed=SHUFFLE_SEED,
    selection="top 1000 leaf topics by unique users",
    **training_stats,
)
_ = (OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")